# 🧪 Quant Lab Setup Guide

This notebook walks through the full setup of your quantitative trading lab environment.

## What's Installed
- **Python 3.12** via `uv` package manager
- **Data Science**: numpy, pandas, scipy, scikit-learn, matplotlib, seaborn, plotly
- **Market Data**: yfinance, pandas-datareader, alpha-vantage
- **Backtesting**: backtrader, zipline-reloaded, quantstats, pyfolio-reloaded
- **Technical Analysis**: ta, ta-lib
- **Machine Learning**: PyTorch, xgboost, lightgbm, stable-baselines3 (RL)
- **Notebooks**: JupyterLab, ipywidgets

## Project Structure
```
quant-lab/
├── notebooks/     # Jupyter notebooks for research
├── strategies/    # Trading strategy implementations
├── data/          # Market data and datasets
├── models/        # Trained ML models
├── backtests/     # Backtest results
└── utils/         # Shared utilities (data_fetcher, indicators, metrics)
```


## 1. Install Required Libraries

All core libraries are already installed via `uv`. Run this cell to verify everything is working:


In [ ]:
# Verify all imports work
import numpy as np
import pandas as pd
import scipy
import sklearn
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import yfinance as yf
import backtrader as bt
import quantstats as qs
import ta
import torch
import xgboost as xgb
import lightgbm as lgb
import gymnasium as gym

print("✅ All libraries imported successfully!")
print(f"  numpy: {np.__version__}")
print(f"  pandas: {pd.__version__}")
print(f"  scikit-learn: {sklearn.__version__}")
print(f"  torch: {torch.__version__}")
print(f"  xgboost: {xgb.__version__}")
print(f"  lightgbm: {lgb.__version__}")


## 2. Configure API Keys

Set up your API keys for OpenAI and Anthropic Claude. Create a `.env` file in the project root:


In [ ]:
import os
from pathlib import Path

# Create .env file if it doesn't exist
env_path = Path(".env")
if not env_path.exists():
    env_content = """# API Keys - DO NOT COMMIT THIS FILE
OPENAI_API_KEY=your_openai_key_here
ANTHROPIC_API_KEY=your_anthropic_key_here
ALPHA_VANTAGE_API_KEY=your_alpha_vantage_key_here

# Trading Configuration
PAPER_TRADING=true
INITIAL_CAPITAL=100000
"""
    env_path.write_text(env_content)
    print("✅ Created .env file — add your API keys there!")
else:
    print("✅ .env file already exists")

# Load environment variables
from dotenv import load_dotenv
load_dotenv()

# Verify keys are set (without printing them)
openai_key = os.getenv("OPENAI_API_KEY", "")
anthropic_key = os.getenv("ANTHROPIC_API_KEY", "")
print(f"OpenAI key set: {'✅' if openai_key and 'your_' not in openai_key else '❌ (not configured)'}")
print(f"Anthropic key set: {'✅' if anthropic_key and 'your_' not in anthropic_key else '❌ (not configured)'}")


## 3. Test Market Data Connectivity

Verify you can fetch market data from Yahoo Finance:


In [ ]:
# Fetch S&P 500 data
tickers = ["SPY", "AAPL", "MSFT", "GOOGL", "AMZN", "TSLA", "NVDA", "META"]
data = yf.download(tickers, start="2023-01-01", end="2025-01-01")["Close"]

print(f"✅ Fetched data for {len(tickers)} tickers")
print(f"   Date range: {data.index[0].date()} to {data.index[-1].date()}")
print(f"   Shape: {data.shape}")
data.tail()


## 4. Add Technical Indicators

Use the `ta` library to compute technical indicators:


In [ ]:
# Fetch OHLCV data for a single stock for indicator calculation
aapl = yf.download("AAPL", start="2023-01-01", end="2025-01-01")

# Add technical indicators using the ta library
from utils.indicators import add_all_indicators
aapl = add_all_indicators(aapl)

print("✅ Technical indicators added:")
print(f"   Columns: {list(aapl.columns)}")
aapl[["Close", "SMA_20", "SMA_50", "RSI", "MACD"]].tail(10)


## 5. Simple Backtest with Backtrader

Run a basic SMA crossover strategy:


In [ ]:
import backtrader as bt
import quantstats as qs
from datetime import datetime

# Define SMA Crossover Strategy
class SmaCross(bt.Strategy):
    params = dict(fast=10, slow=30)

    def __init__(self):
        sma1 = bt.ind.SMA(period=self.p.fast)
        sma2 = bt.ind.SMA(period=self.p.slow)
        self.crossover = bt.ind.CrossOver(sma1, sma2)

    def next(self):
        if not self.position:
            if self.crossover > 0:
                self.buy()
        elif self.crossover < 0:
            self.close()

# Run backtest
cerebro = bt.Cerebro()
cerebro.addstrategy(SmaCross)

# Feed data
data = bt.feeds.PandasData(dataname=yf.download("AAPL", start="2022-01-01", end="2025-01-01"))
cerebro.adddata(data)
cerebro.broker.setcash(100000.0)
cerebro.addsizer(bt.sizers.PercentSizer, percents=95)

print(f"Starting Portfolio Value: ${cerebro.broker.getvalue():,.2f}")
cerebro.run()
print(f"Final Portfolio Value:    ${cerebro.broker.getvalue():,.2f}")
print(f"Total Return:            {(cerebro.broker.getvalue() / 100000 - 1) * 100:.2f}%")


## 6. Performance Metrics with QuantStats

Generate a full performance report:


In [ ]:
# Calculate performance metrics using our utils
from utils.metrics import summary_stats

# Get strategy returns from backtest data
aapl_data = yf.download("AAPL", start="2022-01-01", end="2025-01-01")
returns = aapl_data["Close"].pct_change().dropna()

# Buy and hold stats for comparison
print("📊 Buy & Hold Performance:")
print("=" * 40)
stats = summary_stats(returns)
for key, value in stats.items():
    print(f"  {key}: {value}")

# Plot cumulative returns
cum_returns = (1 + returns).cumprod()
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

cum_returns.plot(ax=axes[0], title="Cumulative Returns - AAPL Buy & Hold")
axes[0].set_ylabel("Cumulative Return")
axes[0].grid(True)

# Drawdown
peak = cum_returns.cummax()
drawdown = (cum_returns - peak) / peak
drawdown.plot(ax=axes[1], title="Drawdown", color="red")
axes[1].set_ylabel("Drawdown")
axes[1].fill_between(drawdown.index, drawdown.values, 0, alpha=0.3, color="red")
axes[1].grid(True)

plt.tight_layout()
plt.show()
